# LIBS Instrument Calibration Report

**Instrument:** <INSTRUMENT_NAME>

This report summarizes the instrument profile calibration process. It compares the measured experimental spectra with a synthetic spectrum generated using a multi-zone plasma model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Print timestamp of report generation
print(f"Report generated on: {time.strftime('%d-%m-%Y %H:%M:%S')}")

In [ ]:
inputCSV = "<INPUT_CSV_PATH>"
input_df = pd.read_csv(inputCSV, header=0, delimiter=';')
wavelengths_as_strings = input_df.columns.to_list()[2:]
input_wavelengths_list = list(map(float, wavelengths_as_strings))

measured_avg_df = pd.read_csv('target_processed.csv', header=0)
wavelengths_raw = measured_avg_df.iloc[:, 0].tolist()
measured_avg_intensities_raw = measured_avg_df.iloc[:, 1].tolist()

# Trimming original measured spectra for representation
startWavelength, endWavelength = wavelengths_raw[0], wavelengths_raw[-1]
startIndex, endIndex = input_wavelengths_list.index(startWavelength), input_wavelengths_list.index(endWavelength)+1
input_wavelengths_list = input_wavelengths_list[startIndex:endIndex]
input_wavelengths = np.array(input_wavelengths_list)

zones_df = pd.read_csv('<ZONES_CSV_PATH>', header=0)
# Extract wavelengths from header (columns 3 onwards: Te, Ne, Weight, w1, w2, ...)
zone_wavelengths_str = zones_df.columns.to_list()[3:]
zone_wavelengths_list = list(map(float, zone_wavelengths_str))
zone_wavelengths = np.array(zone_wavelengths_list)

zones_data = []
for index, row in zones_df.iterrows():
    zones_data.append({
        'Te': row['Te'],
        'Ne': row['Ne'],
        'Weight': row['Weight'],
        'Intensity_Raw': row.iloc[3:].values
    })

In [ ]:
# Interpolate zone spectra to match measured wavelengths
for zone in zones_data:
    zone['Intensity_Interp'] = np.interp(wavelengths_raw, zone_wavelengths, zone['Intensity_Raw'])

In [ ]:
# --- DATA INJECTION SECTION ---
instrument_name = "<INSTRUMENT_NAME>"
wavelengths = np.array(wavelengths_raw)
measured_intensity = np.array(measured_avg_intensities_raw)

# Calculate Combined Spectrum
combined_intensity = np.zeros_like(measured_intensity)

print(f"Calibration Report for: {instrument_name}")
print(f"Optimized Parameters:")

for i, zone in enumerate(zones_data):
    weight = zone['Weight']
    combined_intensity += weight * zone['Intensity_Interp']
    print(f"  Zone {i+1}: Te={zone['Te']:.2f} eV, Ne={zone['Ne']:.2e} cm^-3, Weight={weight:.4f}")

# Scores
rmse = <RMSE>
rsquare_score = <RSQUARE_SCORE>

print(f"R^2: {rsquare_score:.4f}")
print(f"RMSE: {rmse:.4f}")

## 1. Experimental Data Visualization
### 1.1 Raw Measured Spectra (All Shots)

In [ ]:
plt.figure(figsize=(12, 6))
plt.title(f"Raw Measured Spectra - {instrument_name}")
plt.xlabel("Wavelength (nm)")
plt.ylabel("Intensity (a.u.)")

# Plot all shots (limit to first 50 if too many to avoid clutter)
for index, row in input_df.iterrows():
    if index > 50: break
    plt.plot(input_wavelengths, row[startIndex+2:endIndex+2], alpha=0.6, linewidth=0.5)

plt.tight_layout()
plt.show()

### 1.2 Preprocessed Spectrum (Averaged & Baseline Corrected)
This spectrum serves as the target for the calibration optimization.

Baseline correction applied using Asymmetric Least Squares (ALS) with parameters:
* lambda = `<LAMBDA>`
* p = `<P>`  
* maxIterations = `<MAX_ITERATIONS>`

In [ ]:
plt.figure(figsize=(15, 8))
plt.plot(wavelengths, measured_intensity, label='Measured (Avg)', color='black', alpha=0.7, linewidth=1)

plt.title(f'Averaged measured spectrum')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Intensity (a.u.)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Synthetic Zone Spectra
Individual spectra generated for each optimized plasma zone.

In [ ]:
plt.figure(figsize=(15, 6))
for i, zone in enumerate(zones_data):
    label = f"Zone {i+1} (Te={zone['Te']:.2f}eV, Ne={zone['Ne']:.1e})"
    plt.plot(wavelengths, zone['Intensity_Interp'], label=label, alpha=0.7)

plt.title(f'Plasma Zone Contributions')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Intensity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Calibration Result & Validation
Overlay of the measured spectrum with the final weighted synthetic spectrum.

**Formula:** $I_{total} = \sum w_i \cdot I_{zone_i}$

In [ ]:
plt.figure(figsize=(15, 8))
plt.plot(wavelengths, measured_intensity, label='Measured (Avg)', color='black', alpha=0.7, linewidth=1)
plt.plot(wavelengths, combined_intensity, label='Synthetic Combined Multi-Zone Spectrum', color='red', alpha=0.8, linestyle='--')

plt.title(f'Calibration Fit: Measured vs Synthetic (RMSE: {rmse:.4f}, R^2: {rsquare_score:.4f})')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Intensity (a.u.)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Residuals Plot

In [ ]:
# Residuals Plot
residuals = measured_intensity - combined_intensity
plt.figure(figsize=(15, 4))
plt.plot(wavelengths, residuals, color='green', alpha=0.7)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.title('Residuals (Measured - Synthetic)')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Difference')
plt.grid(True, alpha=0.3)
plt.show()